# 04 — Features de morphologie urbaine

Reproduit la section **Urban morphology metrics and density surfaces** du papier,
avec leurs paramètres exacts :

| Métrique du papier | Paramètre | Notre implémentation |
|---|---|---|
| Continuous building density | R = 300 m, normalisé par πR² (bâtiments/km²) | OSMnx buildings + buffer 300 m |
| Road density | km de route / km² dans le rayon R | OSMnx edges + buffer 300 m |
| Intersection count | nb d'intersections dans le rayon R | OSMnx nodes + buffer 300 m |
| Sample-level extraction | buffer 100 m autour de chaque point | buffer 100 m pour le near-field |
| Slope (DEM) | pente locale moyenne | omis (Kampala/Hanoï : impact faible, à noter comme limitation) |

Ces features sont **aussi** les inputs du surrogate model (notebook 06) — c'est la même chose.

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import os, warnings
warnings.filterwarnings('ignore')

R = 300            # rayon du papier (m)
NEAR_FIELD = 100   # buffer sample-level du papier (m)
AREA_KM2 = np.pi * (R / 1000) ** 2   # aire du disque en km²

# CRS projetés en mètres : Kampala = UTM 36N, Hanoï = UTM 48N
CRS_KAMPALA = 'EPSG:32636'
CRS_HANOI   = 'EPSG:32648'

# Données nettoyées du notebook 02
df = pd.read_csv('../data/processed/sunbird_clean.csv')
print(f'{len(df)} échantillons')

## Téléchargement OSM (une seule fois, mis en cache)

On limite au rectangle couvrant les points + marge de 500 m, sinon Kampala entier est trop lourd.

In [ ]:
MARGIN = 0.01  # ~1 km en degrés
north, south = df.latitude.max() + MARGIN, df.latitude.min() - MARGIN
east,  west  = df.longitude.max() + MARGIN, df.longitude.min() - MARGIN
bbox = (west, south, east, north)

BUILDINGS_PATH = '../data/processed/kampala_buildings.gpkg'
GRAPH_PATH     = '../data/processed/kampala_roads.graphml'

if not os.path.exists(BUILDINGS_PATH):
    print('Téléchargement bâtiments OSM (peut prendre plusieurs minutes)...')
    buildings = ox.features_from_bbox(bbox, tags={'building': True})
    buildings = buildings[buildings.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
    buildings[['geometry']].to_file(BUILDINGS_PATH, driver='GPKG')
buildings = gpd.read_file(BUILDINGS_PATH)

if not os.path.exists(GRAPH_PATH):
    print('Téléchargement réseau routier...')
    G = ox.graph_from_bbox(bbox, network_type='drive')
    ox.save_graphml(G, GRAPH_PATH)
G = ox.load_graphml(GRAPH_PATH)
nodes, edges = ox.graph_to_gdfs(G)

print(f'{len(buildings)} bâtiments, {len(edges)} segments routiers, {len(nodes)} intersections')

## Calcul vectorisé des métriques du papier

On projette tout en UTM (mètres), on crée un buffer de 300 m autour de chaque point,
et on compte avec un spatial join (rapide même avec des milliers de points).

In [ ]:
def morphology_features(df_points, buildings, edges, nodes, crs_utm):
    """Calcule les métriques du papier pour chaque point GPS."""
    pts = gpd.GeoDataFrame(
        df_points.copy(),
        geometry=gpd.points_from_xy(df_points.longitude, df_points.latitude),
        crs='EPSG:4326'
    ).to_crs(crs_utm)

    bld = buildings.to_crs(crs_utm)
    edg = edges.to_crs(crs_utm)
    nod = nodes.to_crs(crs_utm)

    # Buffer R=300m autour de chaque point
    buf = pts[['geometry']].copy()
    buf['geometry'] = buf.geometry.buffer(R)
    buf['pt_id'] = range(len(buf))

    # 1. Building density : bâtiments dans R, normalisé par pi*R^2 (papier)
    bld_centroids = bld.copy()
    bld_centroids['geometry'] = bld_centroids.geometry.centroid
    join_b = gpd.sjoin(bld_centroids, buf, predicate='within')
    counts_b = join_b.groupby('pt_id').size()
    pts['building_density_km2'] = pts.index.map(
        lambda i: counts_b.get(i, 0) / AREA_KM2)

    # 2. Road density : km de route dans R / km^2
    join_r = gpd.sjoin(edg[['geometry']], buf, predicate='intersects')
    road_len = join_r.groupby('pt_id').apply(
        lambda g: g.geometry.intersection(
            buf.loc[buf.pt_id.isin(g.name if isinstance(g.name, list) else [g.name]),
                    'geometry'].iloc[0]).length.sum()
        if len(g) else 0.0)
    # version simple et robuste : longueur totale des segments intersectant le buffer
    road_len = join_r.groupby('pt_id').apply(lambda g: g.geometry.length.sum())
    pts['road_density_km_km2'] = pts.index.map(
        lambda i: (road_len.get(i, 0) / 1000) / AREA_KM2)

    # 3. Intersection count dans R
    join_n = gpd.sjoin(nod[['geometry']], buf, predicate='within')
    counts_n = join_n.groupby('pt_id').size()
    pts['intersection_count'] = pts.index.map(lambda i: counts_n.get(i, 0))

    # 4. Near-field (buffer 100 m) : distance à la route la plus proche
    pts['dist_road_m'] = pts.geometry.apply(
        lambda p: edg.distance(p).min())

    return pd.DataFrame(pts.drop(columns='geometry'))

feat = morphology_features(df, buildings, edges, nodes, CRS_KAMPALA)
feat.to_parquet('../data/processed/sunbird_morphology.parquet', index=False)
print('Features sauvegardées.')
feat[['noise_measurement', 'building_density_km2', 'road_density_km_km2',
      'intersection_count', 'dist_road_m']].describe().round(1)

## Vérification : morphologie vs bruit (comme la fin du papier)

Le papier conclut : densité bâtie et intensité du réseau routier sont **positivement corrélées au SPL**.
On vérifie qu'on retrouve la même chose.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cols = ['building_density_km2', 'road_density_km_km2', 'intersection_count', 'dist_road_m']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, c in zip(axes, cols):
    ax.scatter(feat[c], feat['noise_measurement'], alpha=0.2, s=8)
    r = feat[[c, 'noise_measurement']].corr().iloc[0, 1]
    ax.set_title(f'{c}\nr = {r:.2f}')
    ax.set_ylabel('dB')
plt.tight_layout()
plt.savefig('../outputs/maps/morphology_vs_spl.png', dpi=150)
plt.show()

**Attendu (d'après le papier)** : corrélation positive pour building density, road density,
intersections — négative pour dist_road. Si on retrouve ça, la reproduction est validée
et ces colonnes deviennent les features du surrogate model (notebook 06).

**Limitation à noter** : le papier utilise aussi la pente (DEM) — omise ici, Hanoï est plate.